[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/10_Temporal_Knowledge_Graphs.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 深入：时序知识图谱（Temporal Knowledge Graphs）

## 概览

本 notebook 深入讲 **时序知识图谱（TKG）**。静态图记录"是什么"，时序图记录"**何时**是什么"——事实、关系和实体随时间的演化。这类能力对以下场景至关重要：

- **公司历史分析**：追踪并购与高管更替
- **供应链监控**：追产品流动与状态变化
- **金融欺诈检测**：分析交易序列

我们将构建一个科技生态 40 年演化史的富场景。

### 先看主线

02 篇 Phase 5 玩过一次"时间旅行"，这里全面展开。贯穿全篇的核心概念是**两种时间戳**：点事件（`timestamp`——某天收购了）和区间（`valid_from`/`valid_to`——当了二十年 CEO）。七步三段：

**建模（第 1–2 步）—— 备好带时间的数据**

TechCorp 与 InnovateInc 的四十年：创始、产品、并购，实体和关系都带时间元数据；`GraphBuilder(enable_temporal=True)` 建图并索引时间属性。

**问时间（第 3–5 步）—— 三种问法**

3. 第 3 步 时序查询："1990 年的 CEO 是谁？"（时间点）、"Alice 怎么连到 SmartAI？"（跨时间路径）
4. 第 4 步 演化分析：图的指标随时间怎么变
5. 第 5 步 模式检测：自动发现"创立→发布→被收购"这类序列

**管版本与可视化（第 6–7 步）—— 落地**

6. 第 6 步 版本管理：快照与版本对比（呼应 24 篇）
7. 第 7 步 时间线可视化：03 篇 `TemporalVisualizer` 的主战场

### 覆盖的关键组件

1. **`GraphBuilder`（时序模式）**：构建带时间感知属性的 KG
2. **`TemporalGraphQuery`**：时间点、区间与路径查询
3. **`TemporalPatternDetector`**：识别序列与循环模式
4. **`TemporalVersionManager`**：管理快照、对比图状态
5. **`TemporalVisualizer`**：交互式时间线与演化图

**文档**：[API 参考](https://semantica.readthedocs.io/reference/kg/）

## 安装

In [ ]:
# !pip install semantica[all]

In [ ]:
import json
from datetime import datetime
from semantica.kg import GraphBuilder, TemporalGraphQuery, TemporalPatternDetector, TemporalVersionManager
from semantica.visualization import TemporalVisualizer
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

# Ensure consistent output for reproducibility
import random
random.seed(42)

## 第 1 步：场景定义与数据准备

定义"TechCorp"和"InnovateInc"的历史数据集——创始人、产品、最终合并。

**时序属性**：
- 实体有 `founded`、`born`、`released` 日期
- 关系有 `timestamp`（点事件）或 `valid_from`/`valid_to`（区间）

In [ ]:
# 1. Define Entities with Temporal Metadata
entities = [
    # Organizations
    {"id": "org_1", "type": "Organization", "name": "TechCorp", "properties": {"founded": "1980-01-01", "industry": "Hardware"}},
    {"id": "org_2", "type": "Organization", "name": "InnovateInc", "properties": {"founded": "1995-06-15", "industry": "Software"}},
    {"id": "org_3", "type": "Organization", "name": "FutureSystems", "properties": {"founded": "2010-03-10", "industry": "AI"}},
    
    # People
    {"id": "per_1", "type": "Person", "name": "Alice Founder", "properties": {"born": "1955-05-20"}},
    {"id": "per_2", "type": "Person", "name": "Bob Coder", "properties": {"born": "1970-08-12"}},
    {"id": "per_3", "type": "Person", "name": "Charlie CEO", "properties": {"born": "1980-02-28"}},
    
    # Products
    {"id": "prod_1", "type": "Product", "name": "HomePC", "properties": {"released": "1985-11-20"}},
    {"id": "prod_2", "type": "Product", "name": "SoftOS", "properties": {"released": "1998-07-25"}},
    {"id": "prod_3", "type": "Product", "name": "SmartAI", "properties": {"released": "2015-01-10"}}
]

# 2. Define Temporal Relationships
relationships = [
    # Founding Events (Point in time)
    {"source": "per_1", "target": "org_1", "type": "founded", "timestamp": "1980-01-01", "properties": {"timestamp": "1980-01-01"}},
    {"source": "per_2", "target": "org_2", "type": "founded", "timestamp": "1995-06-15", "properties": {"timestamp": "1995-06-15"}},
    
    # Employment (Intervals)
    {"source": "per_1", "target": "org_1", "type": "ceo_of", "valid_from": "1980-01-01", "valid_to": "2000-01-01", "properties": {"role": "CEO"}},
    {"source": "per_3", "target": "org_1", "type": "ceo_of", "valid_from": "2000-01-02", "valid_to": "2023-01-01", "properties": {"role": "CEO"}},
    {"source": "per_2", "target": "org_2", "type": "cto_of", "valid_from": "1995-06-15", "valid_to": "2010-05-01", "properties": {"role": "CTO"}},
    
    # Product Launches
    {"source": "org_1", "target": "prod_1", "type": "launched", "timestamp": "1985-11-20", "properties": {"timestamp": "1985-11-20"}},
    {"source": "org_2", "target": "prod_2", "type": "launched", "timestamp": "1998-07-25", "properties": {"timestamp": "1998-07-25"}},
    {"source": "org_3", "target": "prod_3", "type": "launched", "timestamp": "2015-01-10", "properties": {"timestamp": "2015-01-10"}},
    
    # Corporate Actions
    {"source": "org_1", "target": "org_2", "type": "acquired", "timestamp": "2010-05-01", "properties": {"amount": "$5B", "timestamp": "2010-05-01"}},
    {"source": "org_1", "target": "org_3", "type": "invested_in", "timestamp": "2012-08-15", "properties": {"amount": "$100M", "timestamp": "2012-08-15"}}
]

print(f"Defined {len(entities)} entities and {len(relationships)} temporal relationships.")

## 第 2 步：构建时序图

用 `enable_temporal=True` 的 `GraphBuilder`——指示构建器索引 `timestamp`、`valid_from`、`valid_to` 这些时间属性。

In [ ]:
builder = GraphBuilder(
    enable_temporal=True,
    temporal_granularity="day"  # Can be 'year', 'month', 'day', 'hour'
)

temporal_kg = builder.build(entities, relationships)

# The graph object now contains temporal indices
print("Graph built successfully.")
print(f"Nodes: {len(temporal_kg['entities'])}")
print(f"Edges: {len(temporal_kg['relationships'])}")

## 第 3 步：高级时序查询

用 `TemporalGraphQuery` 问带时间的问题——同一家公司，1990 年和 2015 年的 CEO 不是同一个人；关系要"穿越时间"才能连起来。

In [ ]:
query_engine = TemporalGraphQuery()

# 1. Point-in-Time Query
# "Who was the CEO of TechCorp in 1990?"
ceo_1990 = query_engine.query_at_time(
    temporal_kg,
    query="Find the CEO of TechCorp",
    at_time="1990-06-01"
)
print("CEO in 1990:", [e['id'] for e in ceo_1990.get('entities', [])])

# "Who was the CEO of TechCorp in 2015?"
ceo_2015 = query_engine.query_at_time(
    temporal_kg,
    query="Find the CEO of TechCorp",
    at_time="2015-06-01"
)
print("CEO in 2015:", [e['id'] for e in ceo_2015.get('entities', [])])

# 2. Temporal Path Finding
# "How did Alice (Founder) connect to SmartAI (Product released in 2015)?"
# This requires traversing through time: Alice -> founded TechCorp -> invested in FutureSystems -> launched SmartAI
paths = query_engine.find_temporal_paths(
    graph=temporal_kg,
    source="per_1", # Alice
    target="prod_3", # SmartAI
    start_time="1980-01-01",
    end_time="2020-01-01"
)

print(f"\nFound {len(paths)} temporal paths from Alice to SmartAI.")
for i, path in enumerate(paths):
    print(f"Path {i+1}: {path}")

## 第 4 步：图演化分析

用 `analyze_evolution` 分析图的属性随时间怎么变——数量、多样性、稳定性。

In [ ]:
evolution_stats = query_engine.analyze_evolution(
    temporal_kg,
    start_time="1980-01-01",
    end_time="2025-01-01",
    metrics=["count", "diversity", "stability"]
)

print("\nEvolution Statistics (1980-2025):")
print(f"Total Relationships: {evolution_stats.get('count', 'N/A')}")
print(f"Relationship Diversity: {evolution_stats.get('diversity', 'N/A')}")
print(f"Graph Stability: {evolution_stats.get('stability', 'N/A')}")

## 第 5 步：时序模式检测

用 `TemporalPatternDetector` 自动发现重复结构——序列（A → B → C）或循环。

In [ ]:
detector = TemporalPatternDetector()

# Detect sequential patterns (e.g., Founded -> Launched -> Acquired)
sequences = detector.detect_temporal_patterns(
    temporal_kg,
    pattern_type="sequence",
    min_frequency=1
)

print(f"\nDetected {len(sequences)} sequential patterns.")
for seq in sequences[:3]: # Show top 3
    print(f"Pattern: {seq.get('pattern')}")
    print(f"Support: {seq.get('support')}")

## 第 6 步：版本管理与对比

真实场景里图谱是批量更新的。`TemporalVersionManager` 管理这些版本——24 篇变更管理的时序特化版。

In [ ]:
version_manager = TemporalVersionManager()

# Create explicit versions
v1_1990 = version_manager.create_version(temporal_kg, timestamp="1990-01-01", version_label="v1.0 (Early Days)")
v2_2010 = version_manager.create_version(temporal_kg, timestamp="2010-01-01", version_label="v2.0 (Post-Merger)")

# Compare versions
diff = version_manager.compare_versions(v1_1990, v2_2010)

print(f"\nComparing {v1_1990['label']} vs {v2_2010['label']}:")
print(f"New Entities: {diff.get('entities_added', 0)}")
print(f"New Relationships: {diff.get('relationships_added', 0)}")

## 第 7 步：可视化时间线

最后，`TemporalVisualizer` 让数据活起来——交互式时间线和版本对比图。

In [ ]:
visualizer = TemporalVisualizer()

# 1. Interactive Timeline
# Prepare events for visualization (extract from KG)
def extract_events(graph):
    events = []
    for rel in graph['relationships']:
        # Point events
        if rel.get('timestamp'):
            events.append({
                'timestamp': rel['timestamp'],
                'type': rel['type'],
                'label': f"{rel['source']} -> {rel['target']}",
                'entity': rel['source']
            })
        # Interval events (start)
        if rel.get('valid_from'):
             events.append({
                'timestamp': rel['valid_from'],
                'type': f"{rel['type']} (start)",
                'label': f"{rel['source']} -> {rel['target']}",
                'entity': rel['source']
            })
    return {'events': events}

temporal_data = extract_events(temporal_kg)
timeline_fig = visualizer.visualize_timeline(temporal_data, output="interactive")
# In a notebook, this would render a Plotly figure. 
timeline_fig.show()

# 2. Version History Visualization
history = [
    {"version": "v1.0", "timestamp": "1990-01-01", "changes": "Founding Era"},
    {"version": "v2.0", "timestamp": "2010-01-01", "changes": "Expansion Era"},
    {"version": "v3.0", "timestamp": "2020-01-01", "changes": "AI Era"}
]
history_fig = visualizer.visualize_version_history(history, output="interactive")
history_fig.show()

print("Visualizations generated (render requires Jupyter environment).")

## 小结

### 回顾主线

时序三段走完：**建模**（两种时间戳：点事件与区间）、**问时间**（时间点查询、跨时间路径、演化分析、模式检测——"何时是什么"的三种问法）、**管版本与可视化**（快照对比、时间线）。静态图谱答"是什么"，时序图谱答"何时是、怎么变"——这套工作流是 Semantica 时序智能应用的骨架。

本次深入中，我们：
1. 用时序元数据**建模**了复杂的公司历史
2. 用 `GraphBuilder` **构建**了时间感知的知识图谱
3. **查询**了特定时间切片与区间，重建历史
4. **追踪**了时序路径，理解间接连接
5. **分析**了图的演化指标
6. **管理**了版本并可视化时间线
7. 用 `TemporalVisualizer` **可视化**了数据